In [19]:

import os
import pandas as pd
import sqlite3
import numpy as np
import logging
from sqlalchemy import create_engine



# SETUP


BASE_DIR = r"C:\Users\jakes"
os.chdir(BASE_DIR)

print("Working directory:", os.getcwd())

os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    filename="logs/pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(message)s"
)

logging.info("Pipeline started")



# DATABASE CONNECTION


engine = create_engine(r"sqlite:///C:\Users\jakes\Insurance_project.db")


# LOAD DATA


policy = pd.read_sql("SELECT * FROM policy", engine)
claims = pd.read_sql("SELECT * FROM claims", engine)
customers = pd.read_sql("SELECT * FROM customers", engine)

logging.info("Data loaded")



# FEATURE ENGINEERING 

def build_features(policy, claims, customers):

    df = policy.merge(claims, on="PolicyID", how="left")
    df = df.merge(customers, on="CustomerID", how="left")

    df["claim_count"] = df.groupby("CustomerID")["ClaimsID"].transform("count")
    df["avg_claim"] = df.groupby("CustomerID")["Amount"].transform("mean")



  

    df["avg_claim"] = df["avg_claim"].fillna(0)

    return df


features = build_features(policy, claims, customers)

logging.info("Features created")



# PRICING MODEL 


def run_pricing_model(df):

    df["expected_loss"] = df["claim_count"] * df["avg_claim"]
    df["Amount_to_charge_to_make_profit"] = df["expected_loss"] * 1.3

    # safety fix
    df["expected_loss"] = df["expected_loss"].fillna(0)
    df["Amount_to_charge_to_make_profit"] = df["Amount_to_charge_to_make_profit"].fillna(0)

    return df


pricing_results = run_pricing_model(features)

logging.info("Pricing model completed")



# SIMULATION 


def run_simulation(df):

    df["stress_loss"] = df["expected_loss"] * np.random.uniform(1.1, 1.5, len(df))

    df["stress_loss"] = df["stress_loss"].fillna(0)

    return df


scenario_results = run_simulation(pricing_results)

logging.info("Simulation completed")



# SAVE OUTPUTS


os.makedirs("outputs", exist_ok=True)

pricing_results.to_sql(
    "pricing_results",
    engine,
    if_exists="replace",
    index=False
)

scenario_results.to_csv(
    "outputs/scenario_results.csv",
    index=False
)

logging.info("Outputs saved")



# DISPLAY RESULTS 


pricing_results.head(60)

Working directory: C:\Users\jakes


,PolicyID,CustomerID,VehicleID,StartDate,EndDate,LengthOfPolicy,CoverageType,Excess,Premium,ClaimsID,...,Age,Name,Location,Gender,DatePassedDrivingTest,claim_count,avg_claim,expected_loss,Amount_to_charge_to_make_profit,stress_loss
0,1,1,1,2037-04-30,2040-04-30,3,Fully Comprehensive,315.99,540.92,NaN,...,26,Sarah Lambert,Bradyhaven,F,2021-04-12,0,0.000,0.00,0.000,0.000000
1,2,2,2,2022-04-30,2023-04-30,1,Fully Comprehensive,189.00,354.20,NaN,...,45,Shelly Mendoza,Julieburgh,F,2018-08-06,0,0.000,0.00,0.000,0.000000
2,3,3,3,1966-04-30,1969-04-30,3,"Third party, fire and theft",245.65,360.70,NaN,...,21,Sarah Pollard,Spencemouth,F,2024-11-04,0,0.000,0.00,0.000,0.000000
3,4,3,4,1978-04-30,1981-04-30,3,Third party,290.21,275.62,NaN,...,21,Sarah Pollard,Spencemouth,F,2024-11-04,0,0.000,0.00,0.000,0.000000
4,5,4,5,2013-04-30,2015-04-30,2,"Third party, fire and theft",411.17,1681.88,1.0,...,39,Virginia Thompson,North Lauren,F,2015-10-15,2,14566.565,29133.13,37873.069,37793.727533
5,5,4,5,2013-04-30,2015-04-30,2,"Third party, fire and theft",411.17,1681.88,2.0,...,39,Virginia Thompson,North Lauren,F,2015-10-15,2,14566.565,29133.13,37873.069,36812.258182
6,6,5,6,1973-04-30,1975-04-30,2,Third party,301.41,254.38,NaN,...,28,Cynthia Novak,Grayview,F,2024-02-09,0,0.000,0.00,0.000,0.000000
7,7,5,7,1984-04-30,1987-04-30,3,"Third party, fire and theft",238.13,300.69,NaN,...,28,Cynthia Novak,Grayview,F,2024-02-09,0,0.000,0.00,0.000,0.000000
8,8,6,8,1981-04-30,1982-04-30,1,"Third party, fire and theft",304.14,357.58,NaN,...,25,Robert Ramirez,Port James,M,2023-04-06,0,0.000,0.00,0.000,0.000000
9,9,7,9,2017-04-30,2019-04-30,2,Fully Comprehensive,386.44,4526.81,NaN,...,26,Laurie Lang,Tanyabury,F,2021-12-25,0,0.000,0.00,0.000,0.000000


In [20]:
print("TOTAL POLICIES:", len(pricing_results))
print("AVERAGE PREMIUM:", pricing_results["Premium"].mean() if "Premium" in pricing_results.columns else pricing_results["calculated_premium"].mean())
print("TOTAL EXPECTED LOSS:", pricing_results["expected_loss"].sum())

TOTAL POLICIES: 14173
AVERAGE PREMIUM: 985.387112114584
TOTAL EXPECTED LOSS: 30888107.18


In [21]:
report = pricing_results.groupby("CoverageType").agg({
    "expected_loss": "mean",
    "claim_count": "mean"
}).reset_index()

report.to_csv("outputs/coverage_report.csv", index=False)
report

,CoverageType,expected_loss,claim_count
0,Fully Comprehensive,2774.693946,0.447768
1,Third party,1123.848171,0.449472
2,"Third party, fire and theft",2523.566134,0.446602
